In [24]:
import pandas as pd
import numpy as np

ModuleNotFoundError: No module named 'openpyxl'

In [6]:

orders        = pd.read_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\raw\olist_orders_dataset.csv")
customers     = pd.read_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\raw\olist_customers_dataset.csv")
order_items   = pd.read_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\raw\olist_order_items_dataset.csv")
payments      = pd.read_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\raw\olist_order_payments_dataset.csv")
reviews       = pd.read_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\raw\olist_order_reviews_dataset.csv")
products      = pd.read_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\raw\olist_products_dataset.csv")
sellers       = pd.read_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\raw\olist_sellers_dataset.csv")
category_tr   = pd.read_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\raw\product_category_name_translation.csv")
geolocation   = pd.read_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\raw\olist_geolocation_dataset.csv")

# Size check
tables = {
    "orders": orders, "customers": customers, "order_items": order_items,
    "payments": payments, "reviews": reviews, "products": products,
    "sellers": sellers, "category_tr": category_tr, "geolocation": geolocation,
}
for name, df in tables.items():
    print(f"{name:15s} → {df.shape[0]:>8,} rows | {df.shape[1]} cols")

orders          →   99,441 rows | 8 cols
customers       →   99,441 rows | 5 cols
order_items     →  112,650 rows | 7 cols
payments        →  103,886 rows | 5 cols
reviews         →   99,224 rows | 7 cols
products        →   32,951 rows | 9 cols
sellers         →    3,095 rows | 4 cols
category_tr     →       71 rows | 2 cols
geolocation     → 1,000,163 rows | 5 cols


orders and customers share the exact same 99,441 row count — 1:1 mapping with no orphaned customer records.
order_items (112,650) outnumber orders (99,441), averaging ~1.13 items per order — the bulk of purchases are single-item transactions.
All eight columns in the orders table load as strings, including timestamps — explicit datetime casting will be needed before any time-series work.
reviews (99,224) falls slightly short of orders (99,441), meaning ~217 orders have no review record at all.
geolocation dwarfs every other table at 1M rows — heavy deduplication will be required before it's usable for joins.

Taking sample 10000 rows

In [7]:

def explore_table(name, df):
    print("=" * 60)
    print(f"TABLE: {name}  |  Shape: {df.shape}")
    print("=" * 60)
    print("\n--- First 3 rows ---")
    print(df.head(3))
    print("\n--- Column dtypes ---")
    print(df.dtypes)
    print("\n")

explore_table("orders", orders)

TABLE: orders  |  Shape: (99441, 8)

--- First 3 rows ---
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08-07 15:27:45   
2          2018-08-08 13:50:00           2018-08-17 18:06:29   

  order_estimated_delivery_date  
0           2017-10-18 00:00:00  
1           2018-08-13 00:00:00  
2           2018-09-04 00:00:00  

In [8]:

explore_table("customers", customers)

TABLE: customers  |  Shape: (99441, 5)

--- First 3 rows ---
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  

--- Column dtypes ---
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object




In [9]:

explore_table("geolocation", geolocation)

TABLE: geolocation  |  Shape: (1000163, 5)

--- First 3 rows ---
   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                         1037       -23.545621       -46.639292   
1                         1046       -23.546081       -46.644820   
2                         1046       -23.546129       -46.642951   

  geolocation_city geolocation_state  
0        sao paulo                SP  
1        sao paulo                SP  
2        sao paulo                SP  

--- Column dtypes ---
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object




In [10]:


explore_table("order_items", order_items)

TABLE: order_items  |  Shape: (112650, 7)

--- First 3 rows ---
                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   

                         product_id                         seller_id  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   
1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36   
2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d   

   shipping_limit_date  price  freight_value  
0  2017-09-19 09:45:35   58.9          13.29  
1  2017-05-03 11:05:13  239.9          19.93  
2  2018-01-18 14:48:30  199.0          17.87  

--- Column dtypes ---
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float6

In [11]:
explore_table("payments", payments)

TABLE: payments  |  Shape: (103886, 5)

--- First 3 rows ---
                           order_id  payment_sequential payment_type  \
0  b81ef226f3fe1789b1e8b2acac839d17                   1  credit_card   
1  a9810da82917af2d9aefd1278f1dcfa0                   1  credit_card   
2  25e8ea4e93396b6fa0d3dd708e76c1bd                   1  credit_card   

   payment_installments  payment_value  
0                     8          99.33  
1                     1          24.39  
2                     1          65.71  

--- Column dtypes ---
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object




In [12]:
explore_table("review", reviews)

TABLE: review  |  Shape: (99224, 7)

--- First 3 rows ---
                          review_id                          order_id  \
0  7bc2406110b926393aa56f80a40eba40  73fc7af87114b39712e6da79b0a377eb   
1  80e641a11e56f04c1ad469d5645fdfde  a548910a1c6147796b98fdf73dbeba33   
2  228ce5500dc1d8e020d8d1322874b6f0  f9e4b658b201a9f2ecdecbb34bed034b   

   review_score review_comment_title review_comment_message  \
0             4                  NaN                    NaN   
1             5                  NaN                    NaN   
2             5                  NaN                    NaN   

  review_creation_date review_answer_timestamp  
0  2018-01-18 00:00:00     2018-01-18 21:46:59  
1  2018-03-10 00:00:00     2018-03-11 03:05:13  
2  2018-02-17 00:00:00     2018-02-18 14:36:24  

--- Column dtypes ---
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creat

In [13]:
explore_table("products", sellers)

TABLE: products  |  Shape: (3095, 4)

--- First 3 rows ---
                          seller_id  seller_zip_code_prefix     seller_city  \
0  3442f8959a84dea7ee197c632cb2df15                   13023        campinas   
1  d1b65fc7debc3361ea86b5f14c68d2e2                   13844      mogi guacu   
2  ce3ad9de960102d0677a81f5d0bb7b2d                   20031  rio de janeiro   

  seller_state  
0           SP  
1           SP  
2           RJ  

--- Column dtypes ---
seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object




In [14]:
explore_table("sellers", sellers)

TABLE: sellers  |  Shape: (3095, 4)

--- First 3 rows ---
                          seller_id  seller_zip_code_prefix     seller_city  \
0  3442f8959a84dea7ee197c632cb2df15                   13023        campinas   
1  d1b65fc7debc3361ea86b5f14c68d2e2                   13844      mogi guacu   
2  ce3ad9de960102d0677a81f5d0bb7b2d                   20031  rio de janeiro   

  seller_state  
0           SP  
1           SP  
2           RJ  

--- Column dtypes ---
seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object




In [15]:
explore_table("category_tr", category_tr)

TABLE: category_tr  |  Shape: (71, 2)

--- First 3 rows ---
    product_category_name product_category_name_english
0            beleza_saude                 health_beauty
1  informatica_acessorios         computers_accessories
2              automotivo                          auto

--- Column dtypes ---
product_category_name            str
product_category_name_english    str
dtype: object




In [16]:
orders_s = orders.sample(n=10000, random_state = 42).reset_index(drop=True)

order_ids = set(orders_s["order_id"])
customer_ids = set(orders_s["customer_id"])

customers_s = customers[customers["customer_id"].isin(customer_ids)].copy()
order_items_s = order_items[order_items["order_id"].isin(order_ids)].copy()
payments_s = payments[payments["order_id"].isin(order_ids)].copy()
reviews_s = reviews[reviews["order_id"].isin(order_ids)].copy()

product_ids = set(order_items_s["product_id"])
products_s = products[products["product_id"].isin(product_ids)].copy()

seller_ids = set(order_items_s["seller_id"])
sellers_s = sellers[sellers["seller_id"].isin(seller_ids)].copy()

category_tr_s = category_tr.copy()

zip_codes = set(customers_s["customer_zip_code_prefix"]) | set(sellers_s["seller_zip_code_prefix"])
geolocation_s = geolocation[geolocation["geolocation_zip_code_prefix"].isin(zip_codes)].copy()

sampled= {
    "orders" : orders_s, "customers" : customers_s, "order_items" : order_items_s,
    "payments": payments_s, "reviews": reviews_s, "products": products_s,
    "sellers": sellers_s, "category_tr": category_tr_s, "geolocation": geolocation_s,
}

print("AFTER SAMPLING:")
for name, df in sampled.items():
    print(f"{name:15s} -> {df.shape[0]:>7,} rows")





AFTER SAMPLING:
orders          ->  10,000 rows
customers       ->  10,000 rows
order_items     ->  11,383 rows
payments        ->  10,476 rows
reviews         ->   9,959 rows
products        ->   6,747 rows
sellers         ->   1,654 rows
category_tr     ->      71 rows
geolocation     -> 650,746 rows


##Geo Dedupe

In [17]:
def most_frequent(series):
    m = series.mode()
    return m.iloc[0] if not m.empty else np.nan

geo_clean = (
    geolocation_s
    .groupby("geolocation_zip_code_prefix")
    .agg(
        geolocation_lat = ("geolocation_lat", "mean"),
        gelocation_lng = ("geolocation_lng", "mean"),
        geolocation_city = ("geolocation_city", most_frequent),
        geolocation_state = ("geolocation_state", most_frequent),
    )
    .reset_index()
)

print("Before:", geolocation_s.shape[0], "rows")
print("After:", geo_clean.shape[0], "rows (1 per zip)")
geo_clean.head()

    

Before: 650746 rows
After: 6486 rows (1 per zip)


,geolocation_zip_code_prefix,geolocation_lat,gelocation_lng,geolocation_city,geolocation_state
0,1001,-23.550190,-46.634024,sao paulo,SP
1,1005,-23.549456,-46.636733,sao paulo,SP
2,1006,-23.550102,-46.636137,sao paulo,SP
3,1007,-23.550046,-46.637251,sao paulo,SP
4,1008,-23.546002,-46.635886,sao paulo,SP


650K raw geolocation rows collapse to just 6,486 unique zip codes — each zip averaged ~100 duplicate coordinate entries in the source data.

In [18]:
sampled = {
    "orders": orders_s, "customers": customers_s, "order_items": order_items_s,
    "payments": payments_s, "reviews": reviews_s, "products": products_s,
    "sellers": sellers_s, "category_tr": category_tr_s, "geolocation": geo_clean,
}
print("AFTER SAMPLING:")
for name, df in sampled.items():
    print(f"{name:15s} → {df.shape[0]:>7,} rows")

AFTER SAMPLING:
orders          →  10,000 rows
customers       →  10,000 rows
order_items     →  11,383 rows
payments        →  10,476 rows
reviews         →   9,959 rows
products        →   6,747 rows
sellers         →   1,654 rows
category_tr     →      71 rows
geolocation     →   6,486 rows


In [21]:

save_folder = r"D:\projects\Brazilian E-Commerce Public Dataset\data\processed 10000"  

for name, df in sampled.items():
    path = save_folder + "\\" + name + ".csv"
    df.to_csv(path, index=False)
    print("Saved →", path)

Saved → D:\projects\Brazilian E-Commerce Public Dataset\data\processed 10000\orders.csv
Saved → D:\projects\Brazilian E-Commerce Public Dataset\data\processed 10000\customers.csv
Saved → D:\projects\Brazilian E-Commerce Public Dataset\data\processed 10000\order_items.csv
Saved → D:\projects\Brazilian E-Commerce Public Dataset\data\processed 10000\payments.csv
Saved → D:\projects\Brazilian E-Commerce Public Dataset\data\processed 10000\reviews.csv
Saved → D:\projects\Brazilian E-Commerce Public Dataset\data\processed 10000\products.csv
Saved → D:\projects\Brazilian E-Commerce Public Dataset\data\processed 10000\sellers.csv
Saved → D:\projects\Brazilian E-Commerce Public Dataset\data\processed 10000\category_tr.csv
Saved → D:\projects\Brazilian E-Commerce Public Dataset\data\processed 10000\geolocation.csv


\\Data Dictionary

In [22]:
rows = []
for tname, df in tables.items():
    for col in df.columns:
        rows.append({
            "table_name": tname,
            "column_name": col,
            "data_type": str(df[col].dtype),
            "non_null_count": df[col].notna().sum(),
            "missing_pct": round(df[col].isna().mean() * 100, 2),
            "unique_values": df[col].nunique(),
            "sample_value": df[col].dropna().iloc[0] if df[col].notna().any() else "",
            "description": ""   # business meaning — keezha fill pannuvom
        })

data_dict = pd.DataFrame(rows)
print("Total columns documented:", len(data_dict))
data_dict.head(10)


Total columns documented: 52


,table_name,column_name,data_type,non_null_count,missing_pct,unique_values,sample_value,description
0,orders,order_id,str,99441,0.00,99441,e481f51cbdc54678b7cc49136f2d6af7,
1,orders,customer_id,str,99441,0.00,99441,9ef432eb6251297304e76186b10a928d,
2,orders,order_status,str,99441,0.00,8,delivered,
3,orders,order_purchase_timestamp,str,99441,0.00,98875,2017-10-02 10:56:33,
4,orders,order_approved_at,str,99281,0.16,90733,2017-10-02 11:07:15,
5,orders,order_delivered_carrier_date,str,97658,1.79,81018,2017-10-04 19:55:00,
6,orders,order_delivered_customer_date,str,96476,2.98,95664,2017-10-10 21:25:13,
7,orders,order_estimated_delivery_date,str,99441,0.00,459,2017-10-18 00:00:00,
8,customers,customer_id,str,99441,0.00,99441,06b8999e2fba1a1fbc88172c00ba8bc7,
9,customers,customer_unique_id,str,99441,0.00,96096,861eff4711a542e4b93843c6dd7febb0,


In [25]:

descriptions = {
    "order_id": "Unique ID for each order (primary key)",
    "customer_id": "Links order to a customer (one per order)",
    "customer_unique_id": "Real customer identity across multiple orders",
    "order_status": "Order state: delivered, shipped, canceled, etc.",
    "order_purchase_timestamp": "When the customer placed the order",
    "order_approved_at": "When payment was approved",
    "order_delivered_carrier_date": "When order was handed to courier",
    "order_delivered_customer_date": "When order reached the customer",
    "order_estimated_delivery_date": "Promised delivery date",
    "customer_zip_code_prefix": "Customer location zip code",
    "customer_city": "Customer city",
    "customer_state": "Customer state (e.g. SP, RJ)",
    "order_item_id": "Item sequence number within an order",
    "product_id": "Unique product identifier",
    "seller_id": "Unique seller identifier",
    "shipping_limit_date": "Deadline for seller to ship",
    "price": "Product price",
    "freight_value": "Shipping cost",
    "payment_type": "credit_card, boleto, voucher, debit_card",
    "payment_installments": "Number of EMI installments",
    "payment_value": "Amount paid",
    "review_score": "Customer rating 1 to 5",
    "product_category_name": "Product category (Portuguese)",
    "product_category_name_english": "Product category (English)",
    "product_weight_g": "Product weight in grams",
    "geolocation_lat": "Latitude of zip code",
    "geolocation_lng": "Longitude of zip code",
    "seller_city": "Seller city",
    "seller_state": "Seller state",
}

# Map descriptions onto the dictionary
data_dict["description"] = data_dict["column_name"].map(descriptions).fillna("")

# Save as CSV 
out_folder = r"D:\projects\Brazilian E-Commerce Public Dataset\reports"
data_dict.to_csv(out_folder + r"\data_dictionary.csv", index=False)

print("Data Dictionary saved → reports\\data_dictionary.csv ✓")
print("Columns with description:", (data_dict['description'] != '').sum(), "/", len(data_dict))


data_dict

Data Dictionary saved → reports\data_dictionary.csv ✓
Columns with description: 36 / 52


,table_name,column_name,data_type,non_null_count,missing_pct,unique_values,sample_value,description
0,orders,order_id,str,99441,0.00,99441,e481f51cbdc54678b7cc49136f2d6af7,Unique ID for each order (primary key)
1,orders,customer_id,str,99441,0.00,99441,9ef432eb6251297304e76186b10a928d,Links order to a customer (one per order)
2,orders,order_status,str,99441,0.00,8,delivered,"Order state: delivered, shipped, canceled, etc."
3,orders,order_purchase_timestamp,str,99441,0.00,98875,2017-10-02 10:56:33,When the customer placed the order
4,orders,order_approved_at,str,99281,0.16,90733,2017-10-02 11:07:15,When payment was approved
5,orders,order_delivered_carrier_date,str,97658,1.79,81018,2017-10-04 19:55:00,When order was handed to courier
6,orders,order_delivered_customer_date,str,96476,2.98,95664,2017-10-10 21:25:13,When order reached the customer
7,orders,order_estimated_delivery_date,str,99441,0.00,459,2017-10-18 00:00:00,Promised delivery date
8,customers,customer_id,str,99441,0.00,99441,06b8999e2fba1a1fbc88172c00ba8bc7,Links order to a customer (one per order)
9,customers,customer_unique_id,str,99441,0.00,96096,861eff4711a542e4b93843c6dd7febb0,Real customer identity across multiple orders
